# NutriSense AI — Training Notebook

Run on Kaggle GPU (T4 or P100). All outputs saved to `/kaggle/working/`.

**Before running:**
1. Upload this repo's `model/` folder as a Kaggle dataset named `nutrisense-code`
2. Add Food-101, Khana, DeshiFoodBD datasets to this notebook's inputs
3. Upload your self-scraped images as dataset `nutrisense-scraped`
4. Ensure GPU accelerator is enabled (Settings → Accelerator → GPU T4 x2)

In [ ]:
# ── 0. Install extra dependencies ────────────────────────────────────────
!pip install -q icrawler onnx onnxruntime seaborn scikit-learn

In [ ]:
# ── 1. Paths — adjust dataset names to match your Kaggle inputs ──────────
import os, sys

CODE_DIR    = '/kaggle/input/nutrisense-code/model'
FOOD101_DIR = '/kaggle/input/food101/food-101/images'
KHANA_DIR   = '/kaggle/input/khana-dataset/images'
DESHI_DIR   = '/kaggle/input/deshifoodbd/images'
SCRAPED_DIR = '/kaggle/input/nutrisense-scraped'
PAK_DIR     = '/kaggle/input/pakistani-food-dataset'

UNIFIED_DIR = '/kaggle/working/unified_dataset'
WORK_DIR    = '/kaggle/working'
EVAL_DIR    = '/kaggle/working/evaluation'
CKPT_DIR    = '/kaggle/working/checkpoints'

os.makedirs(UNIFIED_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

sys.path.insert(0, CODE_DIR)
print('Paths ready')

In [ ]:
# ── 2. Merge and curate all datasets ─────────────────────────────────────
!python {CODE_DIR}/curate_classes.py \
    --food101     {FOOD101_DIR} \
    --khana       {KHANA_DIR} \
    --deshi       {DESHI_DIR} \
    --scraped     {SCRAPED_DIR} \
    --pakdataset  {PAK_DIR} \
    --output      {UNIFIED_DIR} \
    --min_images  300

In [ ]:
# ── 3. Dataset stats ──────────────────────────────────────────────────────
import pandas as pd

stats = pd.read_csv(f'{WORK_DIR}/dataset_stats.csv')
print(f'Classes : {len(stats)}')
print(f'Total   : {stats.image_count.sum():,}')
print(f'Min     : {stats.image_count.min()}')
print(f'Mean    : {stats.image_count.mean():.0f}')
stats.sort_values('image_count').head(10)

In [ ]:
# ── 4. Train (two-phase transfer learning) ────────────────────────────────
CLASS_INDEX = f'{WORK_DIR}/class_index.json'

!python {CODE_DIR}/train.py \
    --dataset_dir   {UNIFIED_DIR} \
    --class_index   {CLASS_INDEX} \
    --output_dir    {CKPT_DIR} \
    --batch_size    32 \
    --phase1_epochs 5 \
    --phase2_epochs 15 \
    --patience      3 \
    --num_workers   2

In [ ]:
# ── 5. Pick best checkpoint ───────────────────────────────────────────────
import glob, re

ckpts = glob.glob(f'{CKPT_DIR}/nutrisense_*.pth')
BEST_CKPT = max(ckpts, key=lambda p: float(re.search(r'_(\d+\.\d+)_', p).group(1)))
print(f'Best checkpoint: {BEST_CKPT}')

In [ ]:
# ── 6. Ablation study ─────────────────────────────────────────────────────
!python {CODE_DIR}/ablation.py \
    --dataset_dir {UNIFIED_DIR} \
    --class_index {CLASS_INDEX} \
    --output_dir  {EVAL_DIR} \
    --epochs      8

pd.read_csv(f'{EVAL_DIR}/ablation_results.csv')

In [ ]:
# ── 7. Full evaluation + Grad-CAM samples ────────────────────────────────
!python {CODE_DIR}/evaluate.py \
    --checkpoint      {BEST_CKPT} \
    --dataset_dir     {UNIFIED_DIR} \
    --class_index     {CLASS_INDEX} \
    --output_dir      {EVAL_DIR} \
    --gradcam_samples 20

In [ ]:
# ── 8. Display results ────────────────────────────────────────────────────
import json
from IPython.display import Image as IPImage, display

with open(f'{EVAL_DIR}/results.json') as f:
    r = json.load(f)
print(f"Top-1: {r['top1_accuracy']*100:.2f}%")
print(f"Top-3: {r['top3_accuracy']*100:.2f}%")

display(IPImage(f'{EVAL_DIR}/confusion_matrix.png'))
display(IPImage(f'{EVAL_DIR}/per_class_accuracy.png'))

In [ ]:
# ── 9. Show a few Grad-CAM samples ───────────────────────────────────────
import glob as _glob
for p in _glob.glob(f'{EVAL_DIR}/gradcam_samples/*.png')[:4]:
    display(IPImage(p))

In [ ]:
# ── 10. Export to ONNX ───────────────────────────────────────────────────
!python {CODE_DIR}/export.py \
    --checkpoint {BEST_CKPT} \
    --output     {WORK_DIR}/model.onnx

print('\n=== Download these from Kaggle output ===')
for f in ['model.onnx', 'model_class_index.json', 'class_index.json',
          'dataset_stats.csv',
          'evaluation/ablation_results.csv',
          'evaluation/results.json',
          'evaluation/confusion_matrix.png',
          'evaluation/per_class_accuracy.png']:
    path = f'{WORK_DIR}/{f}'
    exists = '✓' if os.path.exists(path) else '✗'
    print(f'  {exists} {f}')